# Task 1 — Post-Incident Traffic Forecasting with MICN and TimesNet (FAST Version)

Versi ini dibuat untuk mengatasi proses yang terlalu lama pada tahap sample building / time split.

**Task:** predict traffic flow following accident/incident for the next **1, 3, and 6 hours** given **24 hours of historical observations**.

Dataset:

```text
final_hourly_flow_allfeature.csv
```

Model:

```text
MICN-style temporal CNN
TimesNet-style temporal CNN
```

Perbedaan versi FAST:

```text
1. Tidak membangun seluruh sample dataset sekaligus.
2. Sample dibatasi sejak awal dengan MAX_SAMPLES_TOTAL.
3. Incident/Post-Incident samples dipertahankan.
4. Normal/general samples diambil sebagian agar training jauh lebih cepat.
5. Cocok untuk laptop/Jupyter lokal.
```

In [3]:
# ============================================================
# 0. Imports and configuration
# ============================================================

import os
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

DATA_PATH = "/Users/andishahifahmuthahharah/Downloads/Dataset_main/Data/final_hourly_flow_allfeature.csv"

INPUT_HOURS = 24
HORIZONS = [1, 3, 6]
MAX_HORIZON = max(HORIZONS)

TRAIN_RATIO = 0.70
VAL_RATIO = 0.10

BATCH_SIZE = 256
EPOCHS = 10
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-5
PATIENCE = 3

# IMPORTANT SPEED SETTINGS
# Naikkan angka ini kalau laptop kamu kuat.
# Untuk test awal, 80_000 sampai 150_000 biasanya cukup.
MAX_SAMPLES_TOTAL = 120_000

# Ambil semua incident/post-incident sample jika memungkinkan,
# lalu tambahkan normal samples sampai total mencapai MAX_SAMPLES_TOTAL.
KEEP_ALL_INCIDENT_SAMPLES = True

MAPE_THRESHOLD = 10.0
SEED = 42

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

Device: cpu


In [4]:
# ============================================================
# 1. Load data
# ============================================================

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f"File not found: {DATA_PATH}\n\n"
        "Put final_hourly_flow_allfeature.csv in the same folder as this notebook, "
        "or change DATA_PATH to the absolute file path."
    )

df = pd.read_csv(DATA_PATH, low_memory=False)
print("Raw shape:", df.shape)
display(df.head())

Raw shape: (1007400, 24)


,station_id,timestamp,total_flow,wgs84_latitude,wgs84_longitude,road_name,suburb,post_code,device_type,quality_rating,lane_count,road_functional_hierarchy,distance_to_intersection,incident_id,is_major_incident,impact_sequence_hour,precipitation,weather_code,apparent_temperature,temperature_2m,wind_gusts_10m,relative_humidity,incident_count,is_anomaly
0,100001,2025-01-01 00:00:00,13.0,-33.878181,150.924942,Cambridge Street,Canley Heights,2166,Tirtl,4,2,5,80.0,NO_INCIDENT,0,-1.0,0.0,0.0,24.377314,21.45,8.640000,84.35288,0,0
1,100001,2025-01-01 01:00:00,9.0,-33.878181,150.924942,Cambridge Street,Canley Heights,2166,Tirtl,4,2,5,80.0,NO_INCIDENT,0,-1.0,0.0,0.0,24.377314,21.45,8.640000,84.35288,0,0
2,100001,2025-01-01 02:00:00,10.0,-33.878181,150.924942,Cambridge Street,Canley Heights,2166,Tirtl,4,2,5,80.0,NO_INCIDENT,0,-1.0,0.0,1.0,23.952227,21.00,5.040000,86.98510,0,0
3,100001,2025-01-01 03:00:00,6.0,-33.878181,150.924942,Cambridge Street,Canley Heights,2166,Tirtl,4,2,5,80.0,NO_INCIDENT,0,-1.0,0.0,1.0,23.543003,20.65,5.760000,88.87851,0,0
4,100001,2025-01-01 04:00:00,7.0,-33.878181,150.924942,Cambridge Street,Canley Heights,2166,Tirtl,4,2,5,80.0,NO_INCIDENT,0,-1.0,0.0,1.0,23.117048,20.25,6.479999,90.81496,0,0


In [5]:
# ============================================================
# 2. Basic cleaning
# ============================================================

required_cols = ["station_id", "timestamp", "total_flow"]
missing_required = [c for c in required_cols if c not in df.columns]
if missing_required:
    raise ValueError(f"Missing required columns: {missing_required}")

df = df.copy()

df["station_id"] = (
    df["station_id"].astype(str).str.strip().str.replace(r"\.0$", "", regex=True)
)

df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
df["total_flow"] = pd.to_numeric(df["total_flow"], errors="coerce")

df = df.dropna(subset=["station_id", "timestamp", "total_flow"]).copy()

if "is_anomaly" in df.columns:
    df["is_anomaly"] = pd.to_numeric(df["is_anomaly"], errors="coerce").fillna(0).astype(int)
    before = len(df)
    df = df[df["is_anomaly"] == 0].copy()
    print("Removed anomaly rows:", before - len(df))

df = df.sort_values(["station_id", "timestamp"]).reset_index(drop=True)

print("Clean shape:", df.shape)
print("Date range:", df["timestamp"].min(), "to", df["timestamp"].max())
print("Stations:", df["station_id"].nunique())

Removed anomaly rows: 162810
Clean shape: (844590, 24)
Date range: 2025-01-01 00:00:00 to 2025-12-31 23:00:00
Stations: 115


In [6]:
# ============================================================
# 3. Incident flags and dynamic features
# ============================================================

for c, default in [
    ("incident_count", 0),
    ("impact_sequence_hour", -1),
    ("is_major_incident", 0),
]:
    if c not in df.columns:
        df[c] = default
    df[c] = pd.to_numeric(df[c], errors="coerce").fillna(default)

df["is_incident_or_post"] = (
    (df["incident_count"] > 0) |
    (df["impact_sequence_hour"] >= 0)
).astype(int)

df["hour"] = df["timestamp"].dt.hour
df["dayofweek"] = df["timestamp"].dt.dayofweek
df["month"] = df["timestamp"].dt.month
df["is_weekend"] = df["dayofweek"].isin([5, 6]).astype(int)

df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
df["dow_sin"] = np.sin(2 * np.pi * df["dayofweek"] / 7)
df["dow_cos"] = np.cos(2 * np.pi * df["dayofweek"] / 7)

dynamic_candidate_features = [
    "total_flow",
    "incident_count",
    "is_major_incident",
    "impact_sequence_hour",
    "is_incident_or_post",
    "precipitation",
    "weather_code",
    "apparent_temperature",
    "temperature_2m",
    "wind_gusts_10m",
    "relative_humidity",
    "hour_sin",
    "hour_cos",
    "dow_sin",
    "dow_cos",
    "is_weekend",
]

dynamic_features = [c for c in dynamic_candidate_features if c in df.columns]

for c in dynamic_features:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df[dynamic_features] = (
    df.groupby("station_id")[dynamic_features]
      .ffill()
      .bfill()
      .fillna(0)
)

print("Dynamic features:", dynamic_features)
print("Number of features:", len(dynamic_features))
print("Incident/post rows:", int(df["is_incident_or_post"].sum()))

Dynamic features: ['total_flow', 'incident_count', 'is_major_incident', 'impact_sequence_hour', 'is_incident_or_post', 'precipitation', 'weather_code', 'apparent_temperature', 'temperature_2m', 'wind_gusts_10m', 'relative_humidity', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'is_weekend']
Number of features: 16
Incident/post rows: 7571


In [7]:
# ============================================================
# 4. FAST sample index construction
# ============================================================

# Instead of creating all X windows immediately, we first build only the valid indices.
# Then we sample a manageable subset before creating X arrays.

records = []

for station_id, g in df.groupby("station_id", sort=False):
    g = g.sort_values("timestamp").reset_index()
    n = len(g)

    start_idx = INPUT_HOURS - 1
    end_idx = n - MAX_HORIZON - 1

    if end_idx < start_idx:
        continue

    local_indices = np.arange(start_idx, end_idx + 1)

    temp = pd.DataFrame({
        "station_id": station_id,
        "row_index": g.loc[local_indices, "index"].values,
        "local_idx": local_indices,
        "timestamp": g.loc[local_indices, "timestamp"].values,
        "is_incident_anchor": g.loc[local_indices, "is_incident_or_post"].values.astype(int),
    })
    records.append(temp)

sample_index_df = pd.concat(records, ignore_index=True)

print("All possible samples:", len(sample_index_df))
print("Incident anchor samples:", int(sample_index_df["is_incident_anchor"].sum()))
display(sample_index_df.head())

All possible samples: 841255
Incident anchor samples: 7543


,station_id,row_index,local_idx,timestamp,is_incident_anchor
0,100001,23,23,2025-01-01 23:00:00,0
1,100001,24,24,2025-01-02 00:00:00,0
2,100001,25,25,2025-01-02 01:00:00,0
3,100001,26,26,2025-01-02 02:00:00,0
4,100001,27,27,2025-01-02 03:00:00,0


In [8]:
# ============================================================
# 5. FAST sample selection
# ============================================================

rng = np.random.default_rng(SEED)

incident_samples = sample_index_df[sample_index_df["is_incident_anchor"] == 1].copy()
normal_samples = sample_index_df[sample_index_df["is_incident_anchor"] == 0].copy()

if len(sample_index_df) > MAX_SAMPLES_TOTAL:
    if KEEP_ALL_INCIDENT_SAMPLES:
        keep_incident = incident_samples
        remaining = max(MAX_SAMPLES_TOTAL - len(keep_incident), 0)

        if remaining > 0:
            keep_normal_idx = rng.choice(normal_samples.index.values, size=min(remaining, len(normal_samples)), replace=False)
            keep_normal = normal_samples.loc[keep_normal_idx]
            sample_index_df = pd.concat([keep_incident, keep_normal], ignore_index=True)
        else:
            keep_incident_idx = rng.choice(incident_samples.index.values, size=MAX_SAMPLES_TOTAL, replace=False)
            sample_index_df = incident_samples.loc[keep_incident_idx].copy()
    else:
        keep_idx = rng.choice(sample_index_df.index.values, size=MAX_SAMPLES_TOTAL, replace=False)
        sample_index_df = sample_index_df.loc[keep_idx].copy()

sample_index_df = sample_index_df.sort_values(["timestamp", "station_id"]).reset_index(drop=True)

print("Selected samples:", len(sample_index_df))
print("Selected incident samples:", int(sample_index_df["is_incident_anchor"].sum()))
display(sample_index_df.head())

Selected samples: 120000
Selected incident samples: 7543


,station_id,row_index,local_idx,timestamp,is_incident_anchor
0,100001,23,23,2025-01-01 23:00:00,0
1,29005,25919,23,2025-01-01 23:00:00,0
2,7179,410787,23,2025-01-01 23:00:00,0
3,7248,463379,23,2025-01-01 23:00:00,0
4,F3FWY005,610818,23,2025-01-01 23:00:00,1


In [9]:
# ============================================================
# 6. Build X and y only for selected samples
# ============================================================

# Create fast access per station.
station_groups = {}
for station_id, g in df.groupby("station_id", sort=False):
    g = g.sort_values("timestamp").reset_index(drop=True)
    station_groups[station_id] = {
        "features": g[dynamic_features].values.astype(np.float32),
        "target": g["total_flow"].values.astype(np.float32),
        "timestamps": g["timestamp"].values,
        "incident": g["is_incident_or_post"].values.astype(int),
    }

X_list = []
y_list = []

for row in sample_index_df.itertuples(index=False):
    station_id = row.station_id
    local_idx = int(row.local_idx)

    sg = station_groups[station_id]
    x_win = sg["features"][local_idx - INPUT_HOURS + 1 : local_idx + 1]
    y_future = [sg["target"][local_idx + h] for h in HORIZONS]

    X_list.append(x_win)
    y_list.append(y_future)

X = np.stack(X_list).astype(np.float32)
y = np.array(y_list, dtype=np.float32)

samples_meta = sample_index_df[["station_id", "timestamp", "is_incident_anchor"]].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Meta shape:", samples_meta.shape)

X shape: (120000, 24, 16)
y shape: (120000, 3)
Meta shape: (120000, 3)


In [10]:
# ============================================================
# 7. Time-based train / validation / test split
# ============================================================

# This should run quickly in the FAST version.
unique_times = np.array(sorted(samples_meta["timestamp"].unique()))
n_times = len(unique_times)

train_end_idx = int(n_times * TRAIN_RATIO)
val_end_idx = int(n_times * (TRAIN_RATIO + VAL_RATIO))

train_end_time = unique_times[train_end_idx - 1]
val_end_time = unique_times[val_end_idx - 1]

samples_meta["split"] = np.where(
    samples_meta["timestamp"] <= train_end_time,
    "train",
    np.where(samples_meta["timestamp"] <= val_end_time, "val", "test")
)

train_idx = np.where(samples_meta["split"].values == "train")[0]
val_idx = np.where(samples_meta["split"].values == "val")[0]
test_idx = np.where(samples_meta["split"].values == "test")[0]

print("Train end:", train_end_time)
print("Validation end:", val_end_time)
print("Train samples:", len(train_idx))
print("Validation samples:", len(val_idx))
print("Test samples:", len(test_idx))

display(pd.crosstab(samples_meta["split"], samples_meta["is_incident_anchor"]))

Train end: 2025-09-18 02:00:00
Validation end: 2025-10-22 22:00:00
Train samples: 84658
Validation samples: 12352
Test samples: 22990


is_incident_anchor,0,1
split,,
test,22062,928
train,78893,5765
val,11502,850


In [11]:
# ============================================================
# 8. Scaling
# ============================================================

num_features = X.shape[-1]

x_scaler = StandardScaler()
y_scaler = StandardScaler()

x_scaler.fit(X[train_idx].reshape(-1, num_features))
X_scaled = x_scaler.transform(X.reshape(-1, num_features)).reshape(X.shape).astype(np.float32)

y_scaler.fit(y[train_idx])
y_scaled = y_scaler.transform(y).astype(np.float32)

print("Scaling complete.")

Scaling complete.


In [12]:
# ============================================================
# 9. Dataset and DataLoader
# ============================================================

class TrafficSeqDataset(Dataset):
    def __init__(self, X_data, y_data, indices):
        self.X_data = X_data
        self.y_data = y_data
        self.indices = np.array(indices)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        idx = self.indices[i]
        return (
            torch.tensor(self.X_data[idx], dtype=torch.float32),
            torch.tensor(self.y_data[idx], dtype=torch.float32),
            int(idx),
        )

train_loader = DataLoader(TrafficSeqDataset(X_scaled, y_scaled, train_idx), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(TrafficSeqDataset(X_scaled, y_scaled, val_idx), batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(TrafficSeqDataset(X_scaled, y_scaled, test_idx), batch_size=BATCH_SIZE, shuffle=False)

print("DataLoaders ready.")

DataLoaders ready.


In [13]:
# ============================================================
# 10. Models
# ============================================================

class MICNStyle(nn.Module):
    def __init__(self, input_dim, output_dim=3, hidden_dim=64, dropout=0.15):
        super().__init__()
        self.branch3 = nn.Conv1d(input_dim, hidden_dim, kernel_size=3, padding=1)
        self.branch5 = nn.Conv1d(input_dim, hidden_dim, kernel_size=5, padding=2)
        self.branch7 = nn.Conv1d(input_dim, hidden_dim, kernel_size=7, padding=3)
        self.proj = nn.Sequential(
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Conv1d(hidden_dim * 3, hidden_dim, kernel_size=1),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, output_dim),
        )

    def forward(self, x):
        x = x.transpose(1, 2)
        z = torch.cat([self.branch3(x), self.branch5(x), self.branch7(x)], dim=1)
        z = self.proj(z)
        z = z.mean(dim=-1)
        return self.head(z)


class TimesBlockStyle(nn.Module):
    def __init__(self, channels, dropout=0.15):
        super().__init__()
        self.conv3 = nn.Conv1d(channels, channels, kernel_size=3, padding=1)
        self.conv5 = nn.Conv1d(channels, channels, kernel_size=5, padding=2)
        self.conv9 = nn.Conv1d(channels, channels, kernel_size=9, padding=4)
        self.mix = nn.Conv1d(channels * 3, channels, kernel_size=1)
        self.norm = nn.BatchNorm1d(channels)
        self.act = nn.GELU()
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        residual = x
        z = torch.cat([self.conv3(x), self.conv5(x), self.conv9(x)], dim=1)
        z = self.mix(z)
        z = self.norm(z)
        z = self.act(z)
        z = self.dropout(z)
        return z + residual


class TimesNetStyle(nn.Module):
    def __init__(self, input_dim, output_dim=3, hidden_dim=64, num_blocks=3, dropout=0.15):
        super().__init__()
        self.embed = nn.Conv1d(input_dim, hidden_dim, kernel_size=1)
        self.blocks = nn.Sequential(*[TimesBlockStyle(hidden_dim, dropout) for _ in range(num_blocks)])
        self.head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, output_dim),
        )

    def forward(self, x):
        x = x.transpose(1, 2)
        z = self.embed(x)
        z = self.blocks(z)
        z = z.mean(dim=-1)
        return self.head(z)

print("Models ready.")

Models ready.


In [14]:
# ============================================================
# 11. Training utilities
# ============================================================

def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, total_n = 0.0, 0

    for xb, yb, _ in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        pred = model(xb)
        loss = criterion(pred, yb)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * xb.size(0)
        total_n += xb.size(0)

    return total_loss / max(total_n, 1)


@torch.no_grad()
def eval_loss(model, loader, criterion):
    model.eval()
    total_loss, total_n = 0.0, 0

    for xb, yb, _ in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        pred = model(xb)
        loss = criterion(pred, yb)

        total_loss += loss.item() * xb.size(0)
        total_n += xb.size(0)

    return total_loss / max(total_n, 1)


def fit_model(model, name):
    model = model.to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    criterion = nn.MSELoss()

    best_val = np.inf
    best_state = None
    wait = 0
    history = []

    for epoch in range(1, EPOCHS + 1):
        tr = train_one_epoch(model, train_loader, optimizer, criterion)
        va = eval_loss(model, val_loader, criterion)
        history.append({"epoch": epoch, "train_loss": tr, "val_loss": va})

        print(f"{name} | epoch {epoch:02d} | train={tr:.6f} | val={va:.6f}")

        if va < best_val:
            best_val = va
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1

        if wait >= PATIENCE:
            print("Early stopping.")
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, pd.DataFrame(history)

In [15]:
# ============================================================
# 12. Train MICN-style
# ============================================================

input_dim = X_scaled.shape[-1]
output_dim = len(HORIZONS)

micn_model = MICNStyle(input_dim=input_dim, output_dim=output_dim)
micn_model, micn_history = fit_model(micn_model, "MICN")

display(micn_history.tail())

MICN | epoch 01 | train=0.568609 | val=0.439622
MICN | epoch 02 | train=0.344898 | val=0.324540
MICN | epoch 03 | train=0.275245 | val=0.286376
MICN | epoch 04 | train=0.239762 | val=0.236544
MICN | epoch 05 | train=0.213811 | val=0.234318
MICN | epoch 06 | train=0.195691 | val=0.206617
MICN | epoch 07 | train=0.185438 | val=0.192343
MICN | epoch 08 | train=0.176269 | val=0.183683
MICN | epoch 09 | train=0.169426 | val=0.174910
MICN | epoch 10 | train=0.163278 | val=0.172455


,epoch,train_loss,val_loss
5,6,0.195691,0.206617
6,7,0.185438,0.192343
7,8,0.176269,0.183683
8,9,0.169426,0.174910
9,10,0.163278,0.172455


In [17]:
# ============================================================
# 13. Train TimesNet-style
# ============================================================

timesnet_model = TimesNetStyle(input_dim=input_dim, output_dim=output_dim)
timesnet_model, timesnet_history = fit_model(timesnet_model, "TimesNet")

display(timesnet_history.tail())

TimesNet | epoch 01 | train=0.348394 | val=0.245865
TimesNet | epoch 02 | train=0.210567 | val=0.195621
TimesNet | epoch 03 | train=0.185388 | val=0.195838
TimesNet | epoch 04 | train=0.172005 | val=0.164136
TimesNet | epoch 05 | train=0.164277 | val=0.174616
TimesNet | epoch 06 | train=0.155734 | val=0.155092
TimesNet | epoch 07 | train=0.153076 | val=0.150344
TimesNet | epoch 08 | train=0.147477 | val=0.154088
TimesNet | epoch 09 | train=0.146909 | val=0.148189
TimesNet | epoch 10 | train=0.144217 | val=0.176179


,epoch,train_loss,val_loss
5,6,0.155734,0.155092
6,7,0.153076,0.150344
7,8,0.147477,0.154088
8,9,0.146909,0.148189
9,10,0.144217,0.176179


In [18]:
# ============================================================
# 14. Prediction
# ============================================================

@torch.no_grad()
def predict_model(model, loader):
    model.eval()
    preds, ys, idxs = [], [], []

    for xb, yb, ib in loader:
        xb = xb.to(DEVICE)
        pred = model(xb).detach().cpu().numpy()
        preds.append(pred)
        ys.append(yb.numpy())
        idxs.append(ib.numpy())

    return np.concatenate(idxs), np.vstack(preds), np.vstack(ys)

test_indices, pred_micn_scaled, y_test_scaled = predict_model(micn_model, test_loader)
_, pred_times_scaled, _ = predict_model(timesnet_model, test_loader)

pred_micn = y_scaler.inverse_transform(pred_micn_scaled)
pred_times = y_scaler.inverse_transform(pred_times_scaled)
y_test_actual = y_scaler.inverse_transform(y_test_scaled)

pred_df = samples_meta.iloc[test_indices].copy().reset_index(drop=True)

for j, h in enumerate(HORIZONS):
    pred_df[f"y_t_plus_{h}"] = y_test_actual[:, j]
    pred_df[f"pred_MICN_t_plus_{h}"] = pred_micn[:, j]
    pred_df[f"pred_TimesNet_t_plus_{h}"] = pred_times[:, j]

display(pred_df.head())

,station_id,timestamp,is_incident_anchor,split,y_t_plus_1,pred_MICN_t_plus_1,pred_TimesNet_t_plus_1,y_t_plus_3,pred_MICN_t_plus_3,pred_TimesNet_t_plus_3,y_t_plus_6,pred_MICN_t_plus_6,pred_TimesNet_t_plus_6
0,29005,2025-10-22 23:00:00,0,test,65.999992,29.063334,-5.091208,56.000004,39.239735,17.331362,164.000000,47.709007,79.444359
1,47024,2025-10-22 23:00:00,0,test,336.000000,54.078991,-2.267355,256.000000,46.662342,25.264925,1163.000000,43.838188,45.535835
2,53004,2025-10-22 23:00:00,0,test,16.999994,155.056564,153.131271,18.000002,178.979187,4.756091,113.999992,574.862366,175.731552
3,7112,2025-10-22 23:00:00,0,test,163.000000,18.137552,9.595194,72.000000,17.221285,2.171892,873.000000,80.203773,85.614037
4,7119-PR,2025-10-22 23:00:00,0,test,121.999992,15.488459,10.483194,-0.000013,12.042727,9.256472,405.000000,29.081078,32.276741


In [19]:
# ============================================================
# 15. Evaluation
# ============================================================

def masked_mape(y_true, y_pred, threshold=MAPE_THRESHOLD):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = y_true > threshold
    if mask.sum() == 0:
        return np.nan
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

def evaluate_predictions(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    valid = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true = y_true[valid]
    y_pred = y_pred[valid]
    if len(y_true) == 0:
        return {"MAE": np.nan, "RMSE": np.nan, "Masked_MAPE": np.nan, "N": 0}
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": mean_squared_error(y_true, y_pred, squared=False),
        "Masked_MAPE": masked_mape(y_true, y_pred),
        "N": len(y_true),
    }

results = []

eval_sets = {
    "General Test": np.ones(len(pred_df), dtype=bool),
    "Incident/Post-Incident Test": pred_df["is_incident_anchor"].eq(1).values,
}

models = {
    "MICN": "pred_MICN_t_plus_{}",
    "TimesNet": "pred_TimesNet_t_plus_{}",
}

for eval_name, mask in eval_sets.items():
    subset = pred_df.loc[mask].copy()
    if subset.empty:
        print(f"Warning: {eval_name} has 0 samples.")
        continue

    for h in HORIZONS:
        target_col = f"y_t_plus_{h}"
        for model_name, template in models.items():
            pred_col = template.format(h)
            metric = evaluate_predictions(subset[target_col], subset[pred_col])
            metric.update({"Eval_Set": eval_name, "Model": model_name, "Horizon": f"t+{h}"})
            results.append(metric)

results_df = pd.DataFrame(results)
results_df = results_df[["Eval_Set", "Model", "Horizon", "MAE", "RMSE", "Masked_MAPE", "N"]]

display(results_df)

pivot_results = results_df.pivot_table(
    index=["Eval_Set", "Model"],
    columns="Horizon",
    values=["MAE", "RMSE", "Masked_MAPE"],
    aggfunc="first"
)

display(pivot_results)

,Eval_Set,Model,Horizon,MAE,RMSE,Masked_MAPE,N
0,General Test,MICN,t+1,62.652954,128.774162,46.079325,22990
1,General Test,TimesNet,t+1,49.458048,104.486834,36.566231,22990
2,General Test,MICN,t+3,90.446239,201.638905,59.905775,22990
3,General Test,TimesNet,t+3,81.304875,181.411705,51.566731,22990
4,General Test,MICN,t+6,123.538183,257.659718,84.403755,22990
5,General Test,TimesNet,t+6,111.336736,239.303018,71.868099,22990
6,Incident/Post-Incident Test,MICN,t+1,111.839058,259.657717,50.337398,928
7,Incident/Post-Incident Test,TimesNet,t+1,88.539271,193.129196,43.418214,928
8,Incident/Post-Incident Test,MICN,t+3,155.918121,336.585077,71.805077,928
9,Incident/Post-Incident Test,TimesNet,t+3,136.040608,277.507252,66.750658,928


MAE                         Masked_MAPE                               RMSE                        
Horizon                                      t+1         t+3         t+6         t+1        t+3         t+6         t+1         t+3         t+6
Eval_Set                    Model                                                                                                              
General Test                MICN       62.652954   90.446239  123.538183   46.079325  59.905775   84.403755  128.774162  201.638905  257.659718
                            TimesNet   49.458048   81.304875  111.336736   36.566231  51.566731   71.868099  104.486834  181.411705  239.303018
Incident/Post-Incident Test MICN      111.839058  155.918121  203.043042   50.337398  71.805077  102.907496  259.657717  336.585077  395.821875
                            TimesNet   88.539271  136.040608  173.146877   43.418214  66.750658  101.773845  193.129196  277.507252  327.964854

In [20]:
# ============================================================
# 16. General vs Incident comparison and save output
# ============================================================

comparison_rows = []

for model_name in results_df["Model"].unique():
    for h in [f"t+{x}" for x in HORIZONS]:
        g = results_df[(results_df["Eval_Set"] == "General Test") &
                       (results_df["Model"] == model_name) &
                       (results_df["Horizon"] == h)]
        i = results_df[(results_df["Eval_Set"] == "Incident/Post-Incident Test") &
                       (results_df["Model"] == model_name) &
                       (results_df["Horizon"] == h)]

        if len(g) == 0 or len(i) == 0:
            continue

        g_mae = float(g["MAE"].iloc[0])
        i_mae = float(i["MAE"].iloc[0])
        g_rmse = float(g["RMSE"].iloc[0])
        i_rmse = float(i["RMSE"].iloc[0])

        comparison_rows.append({
            "Model": model_name,
            "Horizon": h,
            "General_MAE": g_mae,
            "Incident_MAE": i_mae,
            "MAE_Increase": i_mae - g_mae,
            "MAE_Increase_%": ((i_mae - g_mae) / g_mae) * 100 if g_mae != 0 else np.nan,
            "General_RMSE": g_rmse,
            "Incident_RMSE": i_rmse,
            "RMSE_Increase": i_rmse - g_rmse,
            "RMSE_Increase_%": ((i_rmse - g_rmse) / g_rmse) * 100 if g_rmse != 0 else np.nan,
        })

comparison_df = pd.DataFrame(comparison_rows)
display(comparison_df)

#results_df.to_csv("task1_results_MICN_TimesNet_FAST.csv", index=False)
#comparison_df.to_csv("task1_general_vs_incident_comparison_MICN_TimesNet_FAST.csv", index=False)
#pred_df.to_csv("task1_predictions_MICN_TimesNet_FAST.csv", index=False)

#print("Saved output CSV files.")

,Model,Horizon,General_MAE,Incident_MAE,MAE_Increase,MAE_Increase_%,General_RMSE,Incident_RMSE,RMSE_Increase,RMSE_Increase_%
0,MICN,t+1,62.652954,111.839058,49.186104,78.505642,128.774162,259.657717,130.883555,101.638056
1,MICN,t+3,90.446239,155.918121,65.471882,72.387622,201.638905,336.585077,134.946172,66.924670
2,MICN,t+6,123.538183,203.043042,79.504859,64.356506,257.659718,395.821875,138.162157,53.621947
3,TimesNet,t+1,49.458048,88.539271,39.081223,79.018934,104.486834,193.129196,88.642362,84.835916
4,TimesNet,t+3,81.304875,136.040608,54.735734,67.321589,181.411705,277.507252,96.095546,52.970973
5,TimesNet,t+6,111.336736,173.146877,61.810141,55.516394,239.303018,327.964854,88.661835,37.050028


# Interpretation guide

Untuk diskusi seperti paper TraffiDent, bandingkan:

```text
General Test vs Incident/Post-Incident Test
```

Ekspektasi hasil:

```text
Incident/Post-Incident Test error > General Test error
```

Artinya, traffic forecasting setelah incident lebih sulit karena incident membuat pola traffic menjadi tidak reguler.

Jika proses training masih lama, turunkan:

```python
MAX_SAMPLES_TOTAL = 50000
EPOCHS = 5
```

Jika hasil sudah berjalan lancar, kamu bisa naikkan:

```python
MAX_SAMPLES_TOTAL = 200000
EPOCHS = 20
```